In [11]:
#Code sampled from Claude

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.colors import ListedColormap

#assigning numbers for sirs

S, I, R = 0, 1, 2
#assigning colors blue, red, greem

CMAP = ListedColormap(["#1f3fdc", "#d62728", "#2ca02c"])


#making the lattice
def initialize_lattice(L, frac_infected=0.01, rng=None):
    #making a lattice with some exposed fraction (0.01)
    rng = rng or np.random.default_rng()
    lattice = np.full((L, L), S, dtype=np.int8)
    n_infected = max(1, int(round(frac_infected * L * L)))
    #it ensures that 0.01% unique people get infected
    flat_idx = rng.choice(L * L, size=n_infected, replace=False)
    lattice.reshape(-1)[flat_idx] = I
    return lattice


#build the neighborhood, basically
def build_neighbour_tables(L):
    idx = np.arange(L * L).reshape(L, L)
    #a different way to get the neighborhood using np.roll, this gives a neumann neighborhood
    up = np.roll(idx, 1, axis=0).reshape(-1)
    down = np.roll(idx, -1, axis=0).reshape(-1)
    left = np.roll(idx, 1, axis=1).reshape(-1)
    right = np.roll(idx, -1, axis=1).reshape(-1)
    return up, down, left, right


In [16]:


#update the entire lattice together
def sweep(flat_lattice, neighbours, beta, gamma, delta, rng):
    up, down, left, right = neighbours
    N = flat_lattice.size
#i tried to randomize this by picking N random integers, instead of making it go from range O,N
    sites = rng.integers(0, N, size=N)
#i already calculated the probabilities of everything outside the loop, i'll just put it
    rolls = rng.random(N)

    for k in range(N):
        site = sites[k]
        state = flat_lattice[site]
#going from S->I, based on neighbors
        if state == S:
            if (flat_lattice[up[site]] == I or flat_lattice[down[site]] == I or
                    flat_lattice[left[site]] == I or flat_lattice[right[site]] == I):
                if rolls[k] < beta:
                    flat_lattice[site] = I
        #going from I to R, and R to S
        elif state == I:
            if rolls[k] < gamma:
                flat_lattice[site] = R
        else:  # R
            if rolls[k] < delta:
                flat_lattice[site] = S

    return flat_lattice



In [17]:

#running the full simulation, and recording every snapshot (based on snapshot every)
def run_and_record(L, beta, gamma, delta, n_sweeps, frac_infected=0.01,
                    seed=None, snapshot_every=1):
    #generating random numbers based on the seed (to get consistent numbers if need be)
    rng = np.random.default_rng(seed)
    #making the lattice based on the code earlier
    lattice = initialize_lattice(L, frac_infected, rng)
    #making the lattice into a 1d structure (i was trying to make the code faster and easier to work with, and this was suggested by ai)
    flat = lattice.reshape(-1)
    neighbours = build_neighbour_tables(L)
    N = L * L
#making empty arrays for the number of sweeps (so that you can enter values in later)

    S_frac = np.empty(n_sweeps + 1)
    I_frac = np.empty(n_sweeps + 1)
    R_frac = np.empty(n_sweeps + 1)

    def counts():
        return (np.count_nonzero(flat == S) / N,
                np.count_nonzero(flat == I) / N,
                np.count_nonzero(flat == R) / N)

#a copy is made so that future changes don't override it, and the changes are made only in the initial lattice
    snapshots = [lattice.copy()]
    snapshot_times = [0]
#using the definition of counts to update the value of S, E, I, and R (where it will update values every sweep)

    S_frac[0], I_frac[0], R_frac[0] = counts()
#executes all the sweeps from 1 to n_sweeps

    for t in range(1, n_sweeps + 1):
        sweep(flat, neighbours, beta, gamma, delta, rng)
        #add the new fractions of sir per sweep to the lists
        S_frac[t], I_frac[t], R_frac[t] = counts()

        #append every few copies to the list, instead of appending every copy (i.e appending every 100th copy instead of appending every one
        if t % snapshot_every == 0:
            snapshots.append(lattice.copy())
            snapshot_times.append(t)
#calculate the full time (ie number of sweeps)
    time_axis = np.arange(0, n_sweeps + 1)
    return time_axis, S_frac, I_frac, R_frac, snapshot_times, snapshots



In [18]:

#make the animation - a graph with an arrow with a synchronized lattice diagram
def build_animation(L=100, beta=0.32, gamma=0.01, delta=0.001,
                     n_sweeps=10000, seed=1, snapshot_every=5, interval=50):

    time_axis, S_frac, I_frac, R_frac, snap_t, snaps = run_and_record(
        L, beta, gamma, delta, n_sweeps, frac_infected=0.01,
        seed=seed, snapshot_every=snapshot_every,
    )
#2 subplots - one for the graph and one for the lattice
    fig, (ax_ts, ax_grid) = plt.subplots(
        1, 2, figsize=(17, 9), gridspec_kw={"width_ratios": [1.5, 1]}
    )

#the full graph drawn to the left, all the parameters are here
    ax_ts.plot(time_axis, S_frac, color="tab:blue", label="Susceptible")
    ax_ts.plot(time_axis, I_frac, color="tab:red", label="Infected")
    ax_ts.plot(time_axis, R_frac, color="tab:green", label="Recovered")
    ax_ts.set_xlim(0, time_axis[-1])
    ax_ts.set_ylim(0, 1)
    ax_ts.set_xlabel("Time Step")
    ax_ts.set_ylabel("Fraction of Individuals")
    ax_ts.set_title(f"SIRS with beta={beta}, gamma={gamma}, delta={delta}", fontsize=10)
    ax_ts.legend(loc="upper right", fontsize=8)

    vline = ax_ts.axvline(0, color="gray", linestyle="--", linewidth=0.8)

#defining an arrow to the graph to synchronize the lattice with the arrow, and annotating the arrow t move per timeframe (code sampled from ai)
    arrow_holder = {
        "artist": ax_ts.annotate(
            "", xy=(0, 1.0), xytext=(0, 1.18),
            xycoords=("data", "axes fraction"),
            textcoords=("data", "axes fraction"),
            arrowprops=dict(arrowstyle="-|>", color="black", lw=2),
            annotation_clip=False,
        )
    }

    #the lattice animation on the right is made using this, with the 0,1,2,3 values for seirs
    grid_im = ax_grid.imshow(snaps[0], cmap=CMAP, vmin=0, vmax=2, interpolation="nearest")
    ax_grid.set_xticks([])
    ax_grid.set_yticks([])
    #assigning the title - number of sweeps
    title_grid = ax_grid.set_title(f"Sweep {snap_t[0]}", fontsize=10)

    fig.tight_layout()
    #updating the arrow as it moves - also sampled from AI
    def update(frame_idx):
        t = snap_t[frame_idx]

        vline.set_xdata([t, t])

        arrow_holder["artist"].remove()
        arrow_holder["artist"] = ax_ts.annotate(
            "", xy=(t, 1.0), xytext=(t, 1.18),
            xycoords=("data", "axes fraction"),
            textcoords=("data", "axes fraction"),
            arrowprops=dict(arrowstyle="-|>", color="black", lw=2),
            annotation_clip=False,
        )

        grid_im.set_data(snaps[frame_idx])
        title_grid.set_text(f"Sweep {t}")

        return vline, arrow_holder["artist"], grid_im, title_grid
    #making the animation
    ani = animation.FuncAnimation(
        fig, update, frames=len(snaps), interval=interval, blit=False
    )
    plt.close(fig)  # prevents a duplicate static figure being shown too
    return ani


ani = build_animation()
print("Animation built. In a notebook cell call HTML(ani.to_jshtml()) to display it.")

Animation built. In a notebook cell call HTML(ani.to_jshtml()) to display it.


In [ ]:
from IPython.display import HTML
plt.style.use('default')
plt.rcParams['animation.embed_limit'] = 15000.0
HTML(ani.to_jshtml())
# ani.save('SEIRS+graph - 1.mp4', writer='ffmpeg', fps=30)
